# Chapter 12.5 Solution: IP and 2 o'clock Phase Trombones

Ideally, a trombone map can change the phase advance while leaving the other optical functions unchanged. In a real lattice, however, we still need to realize the desired phase advance by changing quadrupole strengths. At the same time, this matching process should keep the IP Twiss functions unchanged. Therefore, this exercise asks us to reproduce the trombone function near the connections from arc 5 and arc 7 to `IP6`: apply the same phase-advance changes, while preserving the Twiss functions at the IP.

At the IP there are four Twiss constraints, `beta_x`, `alpha_x`, `beta_y`, and `alpha_y`, plus two phase-advance constraints, one in each transverse plane. This gives six constraints in total, so at least six knobs are required. For each side we choose seven quadrupole knobs: three matching-section quadrupoles that connect to the IR, plus four IR quadrupoles on the corresponding side of `IP6`.

This notebook contains the solution scaffold for the Chapter 12 exercise. The exercise is split into three pieces:

1. add a 2 o'clock tune trombone to compensate the net tune shift;
2. match the arc 5 side into `IP6`;
3. match the arc 7 side out of `IP6`.

The compact Bmad exercise names the endpoints `END_5` and `END_7`. In this full ESR SciBmad lattice, the corresponding markers and quadrupole choices are examples that can be adjusted after inspecting `../../common/esr-main-18GeV-1IP.jl`.


In [1]:
using SciBmad
using GTPSA
using LinearAlgebra
using Printf

# Compatibility helpers for the generated ESR lattice file.
if !isdefined(Main, :PhaseRef)
    const PhaseRef = PhaseReference
end

if !isdefined(Main, :BeamlineChildRef)
    struct BeamlineChildRef
        beamline_index::Int
    end
end

findchildren(element, ring) = [
    BeamlineChildRef(i) for (i, candidate) in enumerate(ring.line) if candidate === element
]

# This notebook lives in lattices/chapter_12/exercises/. The candidates below
# let it run from this folder, from Ring_Design_Tutorial_SciBmad/, or from the repository root.
ch12_lattice_dir_candidates = [
    joinpath(pwd(), ".."),
    joinpath(pwd(), "lattices", "chapter_12"),
    joinpath(pwd(), "Ring_Design_Tutorial_SciBmad", "lattices", "chapter_12"),
    joinpath(pwd(), "..", "Ring_Design_Tutorial_SciBmad", "lattices", "chapter_12"),
]
ch12_lattice_dir = normpath(first(filter(
    path -> isfile(joinpath(path, "esr-da-opt.jl")),
    ch12_lattice_dir_candidates,
)))

ring = include(joinpath(ch12_lattice_dir, "esr-da-opt.jl"))

function ch12_zero_controls!()
    for name in fieldnames(typeof(CONTROLS))
        setfield!(CONTROLS, name, 0.0)
    end
end

ch12_zero_controls!()


## Common Matching Helpers

The `beta` and `alpha` targets are taken from the unperturbed design optics. For each IP-side match, the residual keeps `beta_1`, `alpha_1`, `beta_2`, and `alpha_2` fixed at `IP6`, while setting the requested horizontal and vertical phase advance.

The response matrix is computed with GTPSA, using the quadrupole knobs as descriptor parameters. Each response matrix has shape `6 x 7`: six residual constraints and seven quadrupole knobs.


In [2]:
const CH12_ZERO_POWER = [0, 0, 0, 0, 0, 0]

ch12_value(x::Real) = Float64(x)
ch12_value(x) = Float64(x[CH12_ZERO_POWER])

ch12_keep_params(x::Real) = x
ch12_keep_params(x) = x[[CH12_ZERO_POWER...,:]]
ch12_parameter_gradient(x) = GTPSA.gradient(x, include_params=true)[7:end]

function ch12_marker_index(marker, ring)
    findchildren(marker, ring)[1].beamline_index
end

function ch12_forward_phase_advance(tw, marker_a, marker_b, ring, plane)
    i0 = ch12_marker_index(marker_a, ring)
    i1 = ch12_marker_index(marker_b, ring)
    phi = plane == 1 ? tw.table.phi_1 : tw.table.phi_2
    s = tw.table.s

    dphi = ch12_value(phi[i1]) - ch12_value(phi[i0])
    if ch12_value(s[i1]) < ch12_value(s[i0])
        dphi += ch12_value(phi[end])
    end
    return dphi
end

function ch12_forward_phase_advance_keep_params(tw, marker_a, marker_b, ring, plane)
    i0 = ch12_marker_index(marker_a, ring)
    i1 = ch12_marker_index(marker_b, ring)
    phi = plane == 1 ? tw.table.phi_1 : tw.table.phi_2
    s = tw.table.s

    dphi = ch12_keep_params(phi[i1]) - ch12_keep_params(phi[i0])
    if ch12_value(s[i1]) < ch12_value(s[i0])
        dphi += ch12_keep_params(phi[end])
    end
    return dphi
end

function ch12_ip_twiss_target(ring; ip_marker=ip6)
    tw0 = twiss(ring)
    ip_idx = ch12_marker_index(ip_marker, ring)
    return (
        beta_x = ch12_value(tw0.table.beta_1[ip_idx]),
        alpha_x = ch12_value(tw0.table.alpha_1[ip_idx]),
        beta_y = ch12_value(tw0.table.beta_2[ip_idx]),
        alpha_y = ch12_value(tw0.table.alpha_2[ip_idx]),
        tune_x = ch12_value(tw0.tunes[1]),
        tune_y = ch12_value(tw0.tunes[2]),
    )
end

function ch12_match_residual(tw, ring, start_marker, stop_marker, target, phase_target; ip_marker=ip6)
    ip_idx = ch12_marker_index(ip_marker, ring)
    return [
        (ch12_value(tw.table.beta_1[ip_idx]) - target.beta_x) / target.beta_x,
        ch12_value(tw.table.alpha_1[ip_idx]) - target.alpha_x,
        (ch12_value(tw.table.beta_2[ip_idx]) - target.beta_y) / target.beta_y,
        ch12_value(tw.table.alpha_2[ip_idx]) - target.alpha_y,
        ch12_forward_phase_advance(tw, start_marker, stop_marker, ring, 1) - phase_target.x,
        ch12_forward_phase_advance(tw, start_marker, stop_marker, ring, 2) - phase_target.y,
    ]
end

function ch12_match_residual_terms(tw, ring, start_marker, stop_marker, target, phase_target; ip_marker=ip6)
    ip_idx = ch12_marker_index(ip_marker, ring)
    return Any[
        (ch12_keep_params(tw.table.beta_1[ip_idx]) - target.beta_x) / target.beta_x,
        ch12_keep_params(tw.table.alpha_1[ip_idx]) - target.alpha_x,
        (ch12_keep_params(tw.table.beta_2[ip_idx]) - target.beta_y) / target.beta_y,
        ch12_keep_params(tw.table.alpha_2[ip_idx]) - target.alpha_y,
        ch12_forward_phase_advance_keep_params(tw, start_marker, stop_marker, ring, 1) - phase_target.x,
        ch12_forward_phase_advance_keep_params(tw, start_marker, stop_marker, ring, 2) - phase_target.y,
    ]
end

function ch12_gtpsa_response_matrix(apply_knobs!, residual_terms_function, x; ring=ring)
    apply_knobs!(x)
    co_info = find_closed_orbit(ring)

    descriptor = Descriptor([1, 1, 1, 1, 1, 1], 2, ones(Int, length(x)), 1)
    p = params(descriptor)
    apply_knobs!(x .+ p)

    tw = twiss(ring; GTPSA_descriptor=descriptor, co_info=co_info)
    residual_terms = residual_terms_function(tw)

    residual = Float64[ch12_value(term) for term in residual_terms]
    response = reduce(vcat, (ch12_parameter_gradient(term)' for term in residual_terms))

    apply_knobs!(x)
    return residual, Matrix{Float64}(response)
end

const CH12_REFERENCE_TARGET = ch12_ip_twiss_target(ring)

function ch12_lm_optimize(residual_and_response_function, residual_function, x0; maxiter=12)
    x = copy(x0)
    lambdas = [1e-6, 1e-4, 1e-2, 1.0, 100.0]
    scales = [1.0, 0.5, 0.25, 0.1]

    for iter in 0:maxiter-1
        residual, response = residual_and_response_function(x)
        norm_now = norm(residual)
        @printf("iter %2d  residual norm = %.9g  cond(R) = %.3e\n", iter, norm_now, cond(response))

        best_x = copy(x)
        best_norm = norm_now

        for lambda in lambdas
            step = -(response' * response + lambda * I) \ (response' * residual)
            for scale in scales
                trial_x = x .+ scale .* step
                trial_norm = norm(residual_function(trial_x))
                if isfinite(trial_norm) && trial_norm < best_norm
                    best_x = trial_x
                    best_norm = trial_norm
                end
            end
        end

        best_norm >= norm_now - 1e-10 && break
        x = best_x
    end

    return x
end


ch12_lm_optimize (generic function with 1 method)

## Part 1: Add the 2 o'clock Tune Trombone

The IP-side phase trombones change the total phase around the ring. The 2 o'clock tune trombone applies the opposite phase so the total tune remains fixed. In the compact Bmad exercise, this is `TR_M`; in this SciBmad lattice we can use one of the existing trombone marker locations as the tune-compensating location.

The current `esr-da-opt.jl` already has an automatic compensating trombone through `DNUX_IP4` and `DNUY_IP4`. Therefore the block below is left commented by default. Uncomment it only when you want to test the convention where the 2 o'clock trombone carries the tune compensation explicitly.


In [3]:
function ch12_apply_ip_side_trombones!(x1, x2, y1, y2)
    CONTROLS.dnux_mlrf_6 = x1
    CONTROLS.dnux_mlrr_6 = x2
    CONTROLS.dnuy_mlrf_6 = y1
    CONTROLS.dnuy_mlrr_6 = y2
    return nothing
end

function ch12_apply_two_oclock_tune_trombone!(x1, x2, y1, y2)
    # Exercise convention: the 2 o'clock trombone cancels the two IP-side shifts.
    CONTROLS.dnux_ip2 = -(x1 + x2)
    CONTROLS.dnuy_ip2 = -(y1 + y2)
    return nothing
end

function ch12_tune_residual(target)
    tw = twiss(ring)
    return [
        ch12_value(tw.tunes[1]) - target.tune_x,
        ch12_value(tw.tunes[2]) - target.tune_y,
    ]
end


ch12_tune_residual (generic function with 1 method)

In [ ]:
# Uncomment this block to run the 2 o'clock tune-trombone convention.

# ch12_zero_controls!()
# x1 = CH12_W_OPTIMIZED_KNOBS_ITER7.TROMBONE_X1
# x2 = CH12_W_OPTIMIZED_KNOBS_ITER7.TROMBONE_X2
# y1 = CH12_W_OPTIMIZED_KNOBS_ITER7.TROMBONE_Y1
# y2 = CH12_W_OPTIMIZED_KNOBS_ITER7.TROMBONE_Y2
#
# ch12_apply_ip_side_trombones!(x1, x2, y1, y2)
# ch12_apply_two_oclock_tune_trombone!(x1, x2, y1, y2)
#
# println("Tune residual after 2 o'clock compensation:")
# ch12_tune_residual(CH12_REFERENCE_TARGET)


## Part 2: Arc 5 Side, `END_5 -> IP6`

The arc 5 side uses upstream-side quadrupoles to keep the IP Twiss fixed while setting the requested phase advance into `IP6`. The list below is a full ESR SciBmad example. If a more exact `END_5` marker is added, replace `CH12_ARC5_PHASE_START` with that marker.


In [4]:
const CH12_ARC5_PHASE_START = mlrf_6
const CH12_ARC5_PHASE_STOP = ip6

const CH12_ARC5_QUADS = [
    q14ef_6, q13ef_6, q12ef_6, q11ef_6,
    qff4_5, qff5_5, qff6_5,
]
const CH12_ARC5_KN1_REF = [ch12_value(q.Kn1) for q in CH12_ARC5_QUADS]

function ch12_apply_arc5_quads!(x)
    for (quad, k1, dx) in zip(CH12_ARC5_QUADS, CH12_ARC5_KN1_REF, x)
        quad.Kn1 = k1 + dx
    end
    return nothing
end

const CH12_ARC5_PHASE_TARGET = begin
    tw0 = twiss(ring)
    (
        x = ch12_forward_phase_advance(tw0, CH12_ARC5_PHASE_START, CH12_ARC5_PHASE_STOP, ring, 1)
            + CH12_W_OPTIMIZED_KNOBS_ITER7.TROMBONE_X1,
        y = ch12_forward_phase_advance(tw0, CH12_ARC5_PHASE_START, CH12_ARC5_PHASE_STOP, ring, 2)
            + CH12_W_OPTIMIZED_KNOBS_ITER7.TROMBONE_Y1,
    )
end

function ch12_arc5_phase_match_residual(x)
    ch12_apply_arc5_quads!(x)
    tw = twiss(ring)
    return ch12_match_residual(
        tw,
        ring,
        CH12_ARC5_PHASE_START,
        CH12_ARC5_PHASE_STOP,
        CH12_REFERENCE_TARGET,
        CH12_ARC5_PHASE_TARGET,
    )
end

function ch12_arc5_phase_match_residual_terms(tw)
    return ch12_match_residual_terms(
        tw,
        ring,
        CH12_ARC5_PHASE_START,
        CH12_ARC5_PHASE_STOP,
        CH12_REFERENCE_TARGET,
        CH12_ARC5_PHASE_TARGET,
    )
end

function ch12_arc5_phase_match_residual_and_response(x)
    ch12_gtpsa_response_matrix(
        ch12_apply_arc5_quads!,
        ch12_arc5_phase_match_residual_terms,
        x,
    )
end

# Uncomment to inspect the 6 x 7 GTPSA response matrix:
# arc5_residual, arc5_response = ch12_arc5_phase_match_residual_and_response(zeros(length(CH12_ARC5_QUADS)))
# @show size(arc5_response)
#
# Uncomment to solve this match using the GTPSA response matrix:
# arc5_solution = ch12_lm_optimize(
#     ch12_arc5_phase_match_residual_and_response,
#     ch12_arc5_phase_match_residual,
#     zeros(length(CH12_ARC5_QUADS)),
# )
# ch12_apply_arc5_quads!(arc5_solution)


ch12_arc5_phase_match_residual (generic function with 1 method)

## Part 3: Arc 7 Side, `IP6 -> END_7`

The arc 7 side uses downstream-side quadrupoles to keep the IP Twiss fixed while setting the requested phase advance out of `IP6`. In this full ESR SciBmad lattice, `marc_end` is used as the `END_7`-like marker in the W-function optimization.


In [5]:
const CH12_ARC7_PHASE_START = ip6
const CH12_ARC7_PHASE_STOP = marc_end

const CH12_ARC7_QUADS = [
    q1er_6, q2er_6, q3er_6, q4er_6,
    q10er_6, q11er_6, q12er_6,
]
const CH12_ARC7_KN1_REF = [ch12_value(q.Kn1) for q in CH12_ARC7_QUADS]

function ch12_apply_arc7_quads!(x)
    for (quad, k1, dx) in zip(CH12_ARC7_QUADS, CH12_ARC7_KN1_REF, x)
        quad.Kn1 = k1 + dx
    end
    return nothing
end

const CH12_ARC7_PHASE_TARGET = begin
    tw0 = twiss(ring)
    (
        x = ch12_forward_phase_advance(tw0, CH12_ARC7_PHASE_START, CH12_ARC7_PHASE_STOP, ring, 1)
            + CH12_W_OPTIMIZED_KNOBS_ITER7.TROMBONE_X2,
        y = ch12_forward_phase_advance(tw0, CH12_ARC7_PHASE_START, CH12_ARC7_PHASE_STOP, ring, 2)
            + CH12_W_OPTIMIZED_KNOBS_ITER7.TROMBONE_Y2,
    )
end

function ch12_arc7_phase_match_residual(x)
    ch12_apply_arc7_quads!(x)
    tw = twiss(ring)
    return ch12_match_residual(
        tw,
        ring,
        CH12_ARC7_PHASE_START,
        CH12_ARC7_PHASE_STOP,
        CH12_REFERENCE_TARGET,
        CH12_ARC7_PHASE_TARGET,
    )
end

function ch12_arc7_phase_match_residual_terms(tw)
    return ch12_match_residual_terms(
        tw,
        ring,
        CH12_ARC7_PHASE_START,
        CH12_ARC7_PHASE_STOP,
        CH12_REFERENCE_TARGET,
        CH12_ARC7_PHASE_TARGET,
    )
end

function ch12_arc7_phase_match_residual_and_response(x)
    ch12_gtpsa_response_matrix(
        ch12_apply_arc7_quads!,
        ch12_arc7_phase_match_residual_terms,
        x,
    )
end

# Uncomment to inspect the 6 x 7 GTPSA response matrix:
# arc7_residual, arc7_response = ch12_arc7_phase_match_residual_and_response(zeros(length(CH12_ARC7_QUADS)))
# @show size(arc7_response)
#
# Uncomment to solve this match using the GTPSA response matrix:
# arc7_solution = ch12_lm_optimize(
#     ch12_arc7_phase_match_residual_and_response,
#     ch12_arc7_phase_match_residual,
#     zeros(length(CH12_ARC7_QUADS)),
# )
# ch12_apply_arc7_quads!(arc7_solution)


ch12_arc7_phase_match_residual (generic function with 1 method)

## Suggested Workflow

Run the three parts in this order:

1. choose the two IP-side phase shifts, usually from the W-function optimization result;
2. apply the 2 o'clock tune compensation convention;
3. solve the arc 5 quadrupole match;
4. solve the arc 7 quadrupole match;
5. rerun the W-function optimization from Section 12.4.

If the W correction changes noticeably after the quadrupole matching, repeat the phase match and W optimization once more.
